# Chạy tự động toàn bộ dự án Dubbing trên Google Colab (1 Click)

Nhấn **Run** ở ô Code duy nhất bên dưới để tự động thực hiện toàn bộ quy trình:
1. Clone repository từ GitHub.
2. Cài đặt FFmpeg, Node.js.
3. Cài đặt Node.js dependencies (`npm install`) và Python dependencies (`pip install`).
4. Tự động chạy `device_register.py` lấy `DUANJU_DEVICE_ID` và `DUANJU_INSTALL_ID` động để tạo file `.env` & `settings.json`.
5. Tự động tải `cloudflared` (hoặc fallback `localtunnel`), khởi chạy Backend FastAPI Server và mở Public URL.

In [ ]:
# -----------------------------------------------------
# Quy trình thiết lập & chạy tự động 1-Click trên Colab
# -----------------------------------------------------
import os
import sys
import json
import time
import re
import platform
import shutil
import threading
import urllib.request
import subprocess
from pathlib import Path

print("🚀 [1/5] Clone Repository từ GitHub...")
# Đảm bảo làm sạch và đứng tại /content trước khi clone
os.chdir("/content")
!rm -rf /content/dubbing
!git clone https://github.com/kinyias/dubbing.git /content/dubbing
os.chdir("/content/dubbing")

print("\n📦 [2/5] Cài đặt System Dependencies (FFmpeg, Node.js)...")
!apt-get update -qq
!apt-get install -y ffmpeg nodejs npm wget -qq

print("\n🐍 [3/5] Cài đặt Node.js & Python dependencies...")
!cd /content/dubbing/backend && npm install
!pip install -r /content/dubbing/requirements.txt

print("\n📲 [4/5] Tự động đăng ký thiết bị lấy Device ID & Install ID động...")
BASE_DIR = Path("/content/dubbing")
backend_path = os.path.abspath("/content/dubbing/backend")
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)
sys.path.insert(0, os.path.join(backend_path, "service", "liushen"))

device_id = ""
install_id = ""
try:
    from service.liushen.device_register import device_register
    reg_res = device_register()
    device_id = reg_res.get("device_id", "")
    install_id = reg_res.get("install_id", "")
    print(f"✅ Đăng ký thiết bị thành công! Device ID: {device_id} | Install ID: {install_id}")
except Exception as e:
    print(f"⚠️ Lỗi đăng ký thiết bị: {e}")

# Tạo file .env
env_content = f"""DUANJU_DEVICE_ID={device_id}
DUANJU_INSTALL_ID={install_id}
DUANJU_PLATFORM=android
APP_PORT=8000
OPEN_BROWSER=0
FLASK_DEBUG=0
FFMPEG_BIN=ffmpeg
"""
with open("/content/dubbing/.env", "w", encoding="utf-8") as f:
    f.write(env_content)

# Tạo file backend/settings.json
settings_example_path = "/content/dubbing/backend/settings.example.json"
settings_path = "/content/dubbing/backend/settings.json"
if os.path.exists(settings_example_path):
    with open(settings_example_path, "r", encoding="utf-8") as f:
        settings = json.load(f)
    settings["ffmpegPath"] = "ffmpeg"
    with open(settings_path, "w", encoding="utf-8") as f:
        json.dump(settings, f, indent=4, ensure_ascii=False)
print("✅ Khởi tạo cấu hình .env và settings.json hoàn tất!")

print("\n🌐 [5/5] Khởi chạy Backend Server & Mở Public Tunnel...")
backend_proc = subprocess.Popen(["python", "/content/dubbing/backend/main.py"])
time.sleep(3)

def start_cloudflare_tunnel(port: int = 8000) -> str:
    """Download cloudflared binary và mở Cloudflare tunnel."""
    system = platform.system().lower()
    machine = platform.machine().lower()

    which_cf = shutil.which("cloudflared") or shutil.which("cloudflared.exe") or "/usr/local/bin/cloudflared"
    if which_cf and os.path.exists(which_cf) and os.access(which_cf, os.X_OK):
        bin_path = Path(which_cf)
    else:
        binary_name = "cloudflared.exe" if system == "windows" else "cloudflared"
        bin_path = BASE_DIR / binary_name

    if not bin_path.exists() or not os.access(bin_path, os.X_OK):
        print(" [Tunnel] Đang tải Cloudflare Tunnel (cloudflared)...", flush=True)
        if system == "windows":
            url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
        elif system == "darwin":
            url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-amd64.tgz"
        elif "arm" in machine or "aarch64" in machine:
            url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64"
        else:
            url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

        try:
            urllib.request.urlretrieve(url, str(bin_path))
            if system != "windows":
                bin_path.chmod(0o755)
        except Exception as exc:
            print(f" Lỗi tải cloudflared: {exc}", flush=True)
            return ""

    cmd = [str(bin_path), "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"]
    tunnel_url = [""]
    url_found_event = threading.Event()

    def _monitor():
        try:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding="utf-8",
                errors="ignore"
            )
            for line in proc.stdout:
                print(line, end="")
                m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line.strip())
                if m:
                    tunnel_url[0] = m.group(0)
                    url_found_event.set()
        except Exception:
            url_found_event.set()

    threading.Thread(target=_monitor, daemon=True).start()
    url_found_event.wait(timeout=25)
    return tunnel_url[0]

def start_localtunnel(port: int = 8000) -> str:
    """Start localtunnel via npx."""
    if not shutil.which("npx"):
        return ""

    cmd = ["npx", "-y", "localtunnel", "--port", str(port)]
    tunnel_url = [""]
    url_found_event = threading.Event()

    def _monitor():
        try:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding="utf-8",
                errors="ignore"
            )
            for line in proc.stdout:
                print(line, end="")
                m = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", line.strip())
                if m:
                    tunnel_url[0] = m.group(0)
                    url_found_event.set()
        except Exception:
            url_found_event.set()

    threading.Thread(target=_monitor, daemon=True).start()
    url_found_event.wait(timeout=15)
    return tunnel_url[0]

print("🚀 Đang khởi chạy Cloudflare Tunnel...")
public_url = start_cloudflare_tunnel(8000)
if not public_url:
    print("⚠️ Cloudflare Tunnel không khả dụng, đang chuyển sang localtunnel...")
    public_url = start_localtunnel(8000)

if public_url:
    print("\n==================================================")
    print(f"🎉 PUBLIC API URL: {public_url}")
    print("==================================================\n")
else:
    print("❌ Không thể tạo Public URL. Vui lòng kiểm tra lại bối cảnh mạng.")